# Caso 1 · Cox07 — edición de estudio
Adaptación didáctica de [A. M. Mayoral, GLM, caso 1](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html). Fuente fijada: [revisión 25-11-2026](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso1).

Este cuaderno conserva el recorrido y los conceptos de las seis unidades, con explicaciones propias y código R adaptado a Cox07. No reproduce literalmente el texto docente. Las cifras y conclusiones deben proceder de la ejecución. El ejemplo GLOW se sustituye por la cohorte simulada del equipo; las respuestas nominal y ordinal son distintas variables, no dos nombres para la misma variable.

In [ ]:
SEMILLA_CURSO <- 20252026L
set.seed(SEMILLA_CURSO)
options(repr.plot.width=11,repr.plot.height=6)
paquetes <- c("ggplot2","dplyr","tidyr","broom","lme4","ordinal","nnet","survival","pROC","DHARMa","car","MASS","logistf")
faltan <- setdiff(paquetes,rownames(installed.packages()))
if(length(faltan)) install.packages(faltan,repos="https://cloud.r-project.org")
stopifnot(all(vapply(paquetes,requireNamespace,logical(1),quietly=TRUE)))
library(dplyr)
library(ggplot2)
library(survival)
theme_set(theme_minimal(base_size=14))
base <- "https://raw.githubusercontent.com/asunmayoral/GLM/2623c88e6633507967a355cd40a8dd403ee90b0f/caso1/datos_equipos/equipo_07/"
leer <- function(nombre) {
  ruta <- tempfile(fileext=".rds")
  download.file(paste0(base,nombre),ruta,mode="wb",quiet=TRUE)
  readRDS(ruta)
}
cohorte <- leer("cohorte.rds")
pp <- leer("pp.rds")
stopifnot(!anyDuplicated(cohorte$id),all(cohorte$evento %in% 0:1),all(cohorte$x2 %in% 0:1),all(cohorte$ever==cohorte$evento),!anyDuplicated(pp[c("id","periodo")]),sum(pp$y)==sum(cohorte$evento),nrow(pp)==sum(cohorte$tiempo))
stopifnot(!is.null(attr(cohorte,"verdad")))
cohorte$centro <- factor(cohorte$centro)
cohorte$clase_nom <- factor(cohorte$clase_nom,levels=c("A","B","C"))
cohorte$sever_ord <- ordered(cohorte$sever_ord,levels=c("Leve","Moderado","Grave"))
pp$centro <- factor(pp$centro,levels=levels(cohorte$centro))
pp$periodo <- factor(pp$periodo,levels=1:6)
pp$t <- as.integer(pp$periodo)
set.seed(SEMILLA_CURSO)
ids <- unlist(lapply(split(cohorte$id,cohorte$centro),function(v) sample(v,max(1,floor(.75*length(v))))))
A <- filter(cohorte,id %in% ids)
V <- filter(cohorte,!id %in% ids)
PA <- filter(pp,id %in% ids)
PV <- filter(pp,!id %in% ids)
print(c(pacientes=nrow(cohorte),fracturas=sum(cohorte$evento),periodos=nrow(pp),centros=nlevels(cohorte$centro)))
print(table(cohorte$tiempo,cohorte$evento))
head(cohorte)
head(pp)

## 1 · Cuando la recta no llega: del dato binario al marco GLM
[Unidad original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u11)

🏵 Una fractura es un resultado binario: Y ∈ {0,1}. Su esperanza es una probabilidad, p. OLS no limita la predicción al intervalo [0,1] y supone una varianza constante. Para Bernoulli, Var(Y)=p(1−p). El GLM separa distribución, predictor lineal y enlace. La logística transforma cualquier η real en p=1/(1+exp(−η)).

In [ ]:
ggplot(A,aes(factor(evento),x1,fill=factor(evento)))+geom_violin()+geom_boxplot(width=.12)+scale_fill_manual(values=c("tomato","gold"))+labs(x="Fractura",y="Fragilidad")
ols <- lm(evento~x1+x2,data=A)
fit <- glm(evento~x1+x2,data=A,family=binomial())
rej <- expand.grid(x1=seq(min(A$x1)-1,max(A$x1)+1,length.out=150),x2=0:1)
rej$OLS <- predict(ols,rej)
rej$GLM <- predict(fit,rej,type="response")
ggplot(tidyr::pivot_longer(rej,c(OLS,GLM),names_to="modelo",values_to="p"),aes(x1,p,colour=modelo))+geom_line(linewidth=1)+facet_wrap(~x2)+geom_hline(yintercept=c(0,1),linetype=2)
plot(fitted(ols),residuals(ols));abline(h=0,col="tomato")
plot(fitted(fit),residuals(fit,type="response"));abline(h=0,col="tomato")
summary(fit)

## 2 · Formulación de los GLM: estimación, inferencia, interpretación y evaluación
[Unidad original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u12)

### 2.1 Familia exponencial
La Bernoulli tiene log-verosimilitud yη−log(1+exp(η)). Derivar la segunda parte da p; derivar otra vez da p(1−p). Esa curvatura mide la información.

### 2.2 Enlaces
Logit: log(p/(1−p)). Probit: Φ⁻¹(p). Cloglog: log(−log(1−p)). Los tres representan probabilidades, pero sus coeficientes tienen escalas distintas.

In [ ]:
ajustes <- lapply(c("logit","probit","cloglog"),function(g) glm(evento~x1+x2,data=A,family=binomial(g)))
names(ajustes) <- c("logit","probit","cloglog")
print(sapply(ajustes,AIC))
curvas <- bind_rows(lapply(names(ajustes),function(g) transform(rej,modelo=g,p=predict(ajustes[[g]],rej,type="response"))))
ggplot(curvas,aes(x1,p,colour=modelo))+geom_line()+facet_wrap(~x2)

### 2.3 Interpretación: odds, probabilidad y efectos marginales
exp(β) es una razón de odds en el modelo logit, no una razón de probabilidades. El efecto de tratamiento en probabilidad se calcula comparando ambas predicciones para la misma paciente. La interacción x1:x2 permite que el efecto cambie con la fragilidad.

In [ ]:
print(exp(cbind(OR=coef(fit),confint.default(fit))))
fit_int <- update(fit,.~x1*x2)
print(anova(fit,fit_int,test="LRT"))
print(AIC(fit,fit_int))
# Selección provisional por AIC, antes de mirar reserva.
m <- if(AIC(fit_int)<AIC(fit)) fit_int else fit
cf0 <- transform(A,x2=0)
cf1 <- transform(A,x2=1)
print(mean(predict(m,cf1,type="response")-predict(m,cf0,type="response")))
rej$p <- predict(m,rej,type="response")
ggplot(rej,aes(x1,p,colour=factor(x2)))+geom_line(linewidth=1)+labs(colour="Tratamiento")

### 2.4 Estimación IWLS
El score es Xᵀ(y−p). El peso W es p(1−p). El paso de Fisher scoring resuelve un problema de mínimos cuadrados ponderados. Repetimos hasta que los coeficientes dejen de cambiar.

In [ ]:
X <- model.matrix(fit)
y <- A$evento
beta <- rep(0,ncol(X))
trayectoria <- matrix(beta,nrow=1)
for(i in 1:100) {
  p <- plogis(drop(X%*%beta))
  w <- pmax(p*(1-p),1e-12)
  nuevo <- beta+solve(crossprod(X,w*X),crossprod(X,y-p))
  trayectoria <- rbind(trayectoria,t(nuevo))
  if(max(abs(nuevo-beta))<1e-9) { beta <- nuevo; break }
  beta <- nuevo
}
print(cbind(IWLS=drop(beta),R=coef(fit)))
print(crossprod(X,y-plogis(drop(X%*%beta))))
matplot(trayectoria,type="l",lty=1,xlab="Iteración",ylab="Coeficiente")

### 2.5 Inferencia y selección
La devianza mide la pérdida respecto al modelo saturado. LRT compara modelos anidados. Wald aproxima la incertidumbre local; el perfil usa la forma de la verosimilitud. AIC y BIC penalizan la complejidad. No interpretar la devianza individual Bernoulli como un contraste χ² de bondad sin replicación adecuada.

In [ ]:
print(anova(update(fit,.~1),fit,test="LRT"))
print(car::Anova(fit,type=2,test.statistic="LR"))
print(cbind(Wald=confint.default(fit),Perfil=suppressMessages(confint(fit))))
print(AIC(fit,fit_int))
print(BIC(fit,fit_int))
bgrid <- seq(coef(fit)["x2"]-4*sqrt(vcov(fit)["x2","x2"]),coef(fit)["x2"]+4*sqrt(vcov(fit)["x2","x2"]),length.out=100)
ll <- sapply(bgrid,function(b) as.numeric(logLik(glm(evento~x1+offset(b*x2),data=A,family=binomial()))))
plot(bgrid,2*(max(ll)-ll),type="l",xlab="Coeficiente tratamiento",ylab="Caída de devianza");abline(h=qchisq(.95,1),col="tomato")

### 2.6 Diagnóstico, calibración y separación
La calibración comprueba si una probabilidad prevista corresponde a la frecuencia observada. DHARMa construye residuos cuantiles simulados. Una separación completa puede hacer divergir la máxima verosimilitud; Firth permite una estimación penalizada. No presumimos separación en Cox07.

In [ ]:
sim <- DHARMa::simulateResiduals(m,n=250)
plot(sim)
print(DHARMa::testUniformity(sim))
print(DHARMa::testDispersion(sim))
print(logistf::logistf(formula(m),data=A))
calibrar <- function(y,p) {
  tibble(y=y,p=p) |> mutate(grupo=ntile(p,10)) |> group_by(grupo) |> summarise(observado=mean(y),previsto=mean(p),n=n(),.groups="drop")
}
pV <- predict(m,V,type="response")
ggplot(calibrar(V$evento,pV),aes(previsto,observado))+geom_abline(slope=1,intercept=0)+geom_point(size=3,colour="tomato")
print(mean((pV-V$evento)^2))

### 2.7 Discriminación y clasificación
ROC estudia sensibilidad frente a falsas alarmas. AUC mide ordenación, no calibración. Elegimos umbral de Youden en ajuste y lo evaluamos en reserva. El riesgo observado durante ventanas distintas no equivale al riesgo a seis años.

In [ ]:
rocA <- pROC::roc(A$evento,predict(m,A,type="response"),quiet=TRUE,direction="<")
umbral <- as.numeric(pROC::coords(rocA,"best",ret="threshold",transpose=FALSE)[1,1])
rocV <- pROC::roc(V$evento,pV,quiet=TRUE,direction="<")
plot(rocV);print(pROC::auc(rocV))
print(table(observado=V$evento,predicho=as.integer(pV>=umbral)))

## 3 · Extensión de la respuesta binaria: binomial y politómica
[Unidad original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u13)

### 3.1 Binomial agrupada
Si n pacientes comparten predictores, la suma de fracturas es binomial. Para agrupar fragilidad continua usamos tramos; comparamos contra Bernoulli con exactamente esos mismos predictores. No confundir la pérdida de información por discretizar con la agrupación.

In [ ]:
A$tramo <- cut(A$x1,breaks=unique(quantile(A$x1,seq(0,1,.25))),include.lowest=TRUE)
agr <- A |> group_by(tramo,x2) |> summarise(si=sum(evento),no=n()-si,.groups="drop")
mb <- glm(cbind(si,no)~tramo+x2,data=agr,family=binomial())
mi <- glm(evento~tramo+x2,data=A,family=binomial())
print(cbind(agrupado=coef(mb),individual=coef(mi)))
print(c(deviance=deviance(mb),Pearson=sum(residuals(mb,type="pearson")^2),gl=df.residual(mb)))
print(pchisq(deviance(mb),df.residual(mb),lower.tail=FALSE))
print(summary(update(mb,family=quasibinomial())))
plot(fitted(mb),residuals(mb,type="pearson"));abline(h=0)
print(agr |> mutate(p=fitted(mb),observado=si/(si+no)))

### 3.2 Nominal: endotipo
El endotipo A, B o C no tiene orden. Usamos A como referencia. Cada categoría tiene su propio predictor lineal y la normalización softmax garantiza que las probabilidades sumen uno.

In [ ]:
nom <- nnet::multinom(clase_nom~x1+x2,data=A,trace=FALSE,Hess=TRUE)
nom0 <- nnet::multinom(clase_nom~1,data=A,trace=FALSE)
print(summary(nom))
print(exp(coef(nom)))
print(anova(nom0,nom,test="Chisq"))
prN <- predict(nom,V,type="probs")
print(table(observado=V$clase_nom,predicho=predict(nom,V)))
print(mean(-log(pmax(prN[cbind(seq_len(nrow(V)),match(V$clase_nom,colnames(prN)))],1e-15))))
pr <- cbind(rej[c("x1","x2")],as.data.frame(predict(nom,rej,type="probs")))
ggplot(tidyr::pivot_longer(pr,c(A,B,C),names_to="endotipo",values_to="p"),aes(x1,p,colour=endotipo))+geom_line()+facet_wrap(~x2)

### 3.3 Ordinal: estadio
Leve < Moderado < Grave. El modelo de odds proporcionales usa logit P(Y≤k)=θ_k−Xβ. Un β positivo desplaza la masa hacia estadios más graves. Contrastamos si un efecto común basta para todos los cortes.

In [ ]:
ord <- ordinal::clm(sever_ord~x1+x2,data=A)
print(summary(ord))
print(ordinal::nominal_test(ord))
ord0 <- ordinal::clm(sever_ord~1,data=A)
print(anova(ord0,ord))
cortes <- lapply(1:2,function(k) glm(I(as.integer(sever_ord)>k)~x1+x2,data=A,family=binomial()))
print(sapply(cortes,coef))
nd <- V[c("x1","x2")]
prO <- predict(ord,newdata=nd,type="prob")$fit
print(table(observado=V$sever_ord,predicho=colnames(prO)[max.col(prO)]))
nom_sever <- nnet::multinom(sever_ord~x1+x2,data=A,trace=FALSE)
print(c(ordinal=AIC(ord),nominal=AIC(nom_sever)))
po <- predict(ord,newdata=rej[c("x1","x2")],type="prob")$fit
ggplot(tidyr::pivot_longer(cbind(rej[c("x1","x2")],as.data.frame(po)),c(Leve,Moderado,Grave),names_to="estadio",values_to="p"),aes(x1,p,colour=estadio))+geom_line()+facet_wrap(~x2)

## 4 · Efectos Aleatorios y Modelos Mixtos
[Unidad original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u14)

### 4.1–4.3 Efecto de clínica
Pooling completo ignora diferencias entre centros. No pooling estima cada clínica por separado. El modelo mixto estima una distribución de efectos de clínica, permitiendo encogimiento. Comparamos intercepto aleatorio y pendiente de fragilidad.

In [ ]:
pool <- glm(evento~x1*x2,data=A,family=binomial())
mixto <- lme4::glmer(evento~x1*x2+(1|centro),data=A,family=binomial())
pend <- lme4::glmer(evento~x1*x2+(1+x1|centro),data=A,family=binomial(),control=lme4::glmerControl(optimizer="bobyqa"))
print(AIC(pool,mixto,pend))
print(lme4::isSingular(pend))
print(lme4::VarCorr(mixto))
sig2 <- as.numeric(lme4::VarCorr(mixto)$centro[1,1])
print(c(ICC_latente_logit=sig2/(sig2+pi^2/3)))
u <- lme4::ranef(mixto,condVar=TRUE)$centro
ee <- sqrt(attr(u,"postVar")[1,1,])
df <- data.frame(centro=rownames(u),u=u[,1],ee=ee)
ggplot(df,aes(reorder(centro,u),u))+geom_pointrange(aes(ymin=u-1.96*ee,ymax=u+1.96*ee))+geom_hline(yintercept=0)+coord_flip()
print(summary(mixto))

### 4.4 Diagnóstico, predicción y credibilidad
Una clínica conocida tiene un efecto estimado. Una clínica típica fija el efecto en cero. La media poblacional integra la distribución de efectos, no equivale a la clínica típica. Evaluamos pacientes de reserva y revisamos convergencia y residuos. El contraste del componente de varianza está en la frontera; el LRT ordinario no es exacto.

In [ ]:
eta <- predict(mixto,V,re.form=NA,type="link")
set.seed(SEMILLA_CURSO)
U <- rnorm(2000,0,sqrt(sig2))
pred <- list(pool=predict(pool,V,type="response"),condicional=predict(mixto,V,type="response"),tipica=plogis(eta),marginal=rowMeans(plogis(outer(eta,U,"+"))))
print(sapply(pred,function(p) c(Brier=mean((V$evento-p)^2),AUC=as.numeric(pROC::auc(pROC::roc(V$evento,p,quiet=TRUE,direction="<"))))))
plot(DHARMa::simulateResiduals(mixto,n=250))
LR <- 2*(as.numeric(logLik(mixto))-as.numeric(logLik(pool)))
print(c(LR=LR,p_aprox=if(LR>0) .5*pchisq(LR,1,lower.tail=FALSE) else 1))
perfil <- data.frame(x1=0,x2=0,centro=levels(A$centro)[1])
eta0 <- predict(mixto,perfil,re.form=NA,type="link")
print(c(tipica=plogis(eta0),marginal=mean(plogis(eta0+U)),quantile(plogis(eta0+U),c(.025,.975))))

### 4.5 Extensiones
La estructura (1+x1|centro) permite correlación entre intercepto y pendiente; (1+x1||centro) la restringe a cero. Un ajuste singular indica que los datos no sostienen toda la complejidad. No se debe conservar una estructura solo por aparecer en la teoría.

In [ ]:
indep <- lme4::glmer(evento~x1*x2+(1+x1||centro),data=A,family=binomial())
print(AIC(mixto,pend,indep))
print(c(intercepto=lme4::isSingular(mixto),pendiente=lme4::isSingular(pend),independiente=lme4::isSingular(indep)))

## 5 · Supervivencia: el tiempo hasta el evento como GLM
[Unidad original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u15)

### 5.1–5.2 Riesgo discreto y Kaplan–Meier
El riesgo h_t es la probabilidad de fractura en el periodo t condicionada a seguir en riesgo. S_t=∏(1−h_j). La censura administrativa no es una ausencia de fractura futura. El RDS persona-periodo contiene las filas realmente observadas.

In [ ]:
print(PA |> group_by(periodo) |> summarise(en_riesgo=n(),fracturas=sum(y),h=mean(y),.groups="drop"))
km <- survfit(Surv(tiempo,evento)~x2,data=A)
plot(km,col=c("tomato","darkgreen"),xlab="Revisión",ylab="Supervivencia sin fractura")
legend("bottomleft",legend=c("Sin tratamiento","Tratamiento"),col=c("tomato","darkgreen"),lty=1)
print(survdiff(Surv(tiempo,evento)~x2,data=A))

### 5.3 GLM de riesgos proporcionales
Con cloglog, log(−log(1−h_t))=α_t+Xβ. El exponencial de β es una razón de riesgos bajo el modelo de riesgo continuo subyacente por intervalos. Comparamos riesgo base constante, tendencia lineal, logarítmica y libre por periodo. El efecto aleatorio de clínica modela fragilidad compartida.

In [ ]:
m_pp <- glm(y~periodo+x1+x2,data=PA,family=binomial("cloglog"))
m_const <- glm(y~x1+x2,data=PA,family=binomial("cloglog"))
m_lin <- glm(y~t+x1+x2,data=PA,family=binomial("cloglog"))
m_log <- glm(y~log(t)+x1+x2,data=PA,family=binomial("cloglog"))
print(AIC(m_const,m_lin,m_log,m_pp))
print(anova(m_const,m_pp,test="LRT"))
print(anova(m_log,m_pp,test="LRT"))
m_frail <- lme4::glmer(y~periodo+x1+x2+(1|centro),data=PA,family=binomial("cloglog"))
print(summary(m_frail))
print(exp(lme4::fixef(m_frail)[c("x1","x2")]))
m_ph <- update(m_pp,.~.+periodo:x2+periodo:x1)
print(anova(m_pp,m_ph,test="LRT"))
plot(DHARMa::simulateResiduals(m_frail,n=250))
print(calibrar(PV$y,predict(m_frail,PV,type="response")))

### Predicción de la supervivencia
Cada perfil se sigue durante seis periodos hipotéticos completos. Multiplicar las probabilidades de no fractura acumuladas produce la supervivencia. No se multiplican directamente las probabilidades de fractura.

In [ ]:
pf <- tidyr::expand_grid(x1=c(-1,0,1),x2=0:1,periodo=factor(1:6,levels=levels(PA$periodo)))
pf$centro <- levels(A$centro)[1]
pf$h <- predict(m_frail,pf,type="response",re.form=NA)
pf <- pf |> group_by(x1,x2) |> arrange(periodo,.by_group=TRUE) |> mutate(S=cumprod(1-h),riesgo=1-S) |> ungroup()
ggplot(pf,aes(as.integer(periodo),riesgo,colour=factor(x2)))+geom_line()+geom_point()+facet_wrap(~x1)+labs(x="Revisión",y="Riesgo acumulado",colour="Tratamiento")
print(filter(pf,periodo==6))

### 5.4 Control de calidad contra el DGP
Solo ahora abrimos la verdad de la cohorte. Los parámetros de la binaria pertenecen al modelo de supervivencia cloglog, por tanto no se comparan directamente con los coeficientes logit del desenlace agregado.

In [ ]:
verdad <- attr(cohorte,"verdad")
v <- verdad$binaria
print(cbind(verdad=v$beta[c("x1","x2")],estimado=lme4::fixef(m_frail)[c("x1","x2")]))
print(c(sigma_real=v$sigma_u,sigma_estimada=sqrt(lme4::VarCorr(m_frail)$centro[1,1])))
base_pred <- data.frame(periodo=factor(1:6,levels=levels(PA$periodo)),x1=0,x2=0,centro=levels(A$centro)[1])
print(data.frame(periodo=1:6,verdad=v$alpha_base,estimado=predict(m_frail,base_pred,type="link",re.form=NA)))
print(verdad$nominal)
print(verdad$ordinal)

## 6 · Estudio de caso, exposición y evaluación
[Encargo original](https://asunmayoral.github.io/GLM/caso1/caso1_binaria.html#sec-u16)

El informe técnico debe responder cinco preguntas: riesgo individual y tratamiento; momento de fractura y revisión; endotipo; estadio; diferencias clínicas y predicción en una clínica nueva. Integrar las unidades en una recomendación coherente, distinguir asociación de causalidad y discutir la censura. Adjuntar reflexión sobre IA, reproducibilidad y contraste con el generador.

### Índice de trazabilidad docente
**Unidad 1.1**
- 1.1 El modelo lineal en GLOW
- 1.2 Por qué OLS no sirve para una respuesta binaria
- 1.3 Los tres componentes de un GLM
- 1.4 GLM en datos binarios

**Unidad 1.2**
- 2.1 Familia exponencial y relación media–varianza
- 2.2 El enlace: canónico y alternativas
- 2.3 Interpretación: del coeficiente a la probabilidad
- 2.4 Estimación: máxima verosimilitud e IWLS
- 2.5 Inferencia, selección y bondad de ajuste
- 2.6 Diagnóstico y validación
- 2.7 Evaluación como clasificador

**Unidad 1.3**
- 3.1 Del dato individual al agrupado: respuesta binomial
- 3.2 Más de dos categorías sin orden: politómica nominal
- 3.3 Categorías ordenadas: el modelo de odds proporcionales

**Unidad 1.4**
- 4.1 Qué son y para qué sirven
- 4.2 El GLMM: formulación
- 4.3 Tres modelos para el efecto de centro
- 4.4 Comparación, evaluación y diagnóstico
- 4.5 Extensiones: estructuras aleatorias más ricas

**Unidad 1.5**
- 5.1 La función de supervivencia y la función de riesgo en tiempo discreto
- 5.2 Kaplan–Meier y riesgos proporcionales
- 5.3 El modelo de riesgos proporcionales como GLM
- 5.4 Validación contra el DGP: ¿recuperamos la verdad?

**Unidad 1.6**
- 6.1 El estudio de caso (entregable)
- 6.2 Checklist de tareas

Los scripts originales están disponibles en [GitHub](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso1/scripts). Esta edición adapta el recorrido; no incluye cada bloque original ni todas sus explicaciones. Pendiente comprobar ejecución completa en R.

In [ ]:
sessionInfo()